In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import box

from wildlife_datasets.datasets import CzechLynxv2

In [ ]:
root = "/data/wildlife_datasets/data/CzechLynx"
dataset = CzechLynxv2(root)

mask = dataset.metadata["real_animal"]
dataset = dataset.get_subset(mask)

In [ ]:
def count_seen(seen: dict, ids: set, n: int) -> int:
    return sum(seen.get(i, 0) == n for i in ids)


def add_location(seen: dict, ids: set) -> None:
    for i in ids:
        seen[i] = seen.get(i, 0) + 1


def greedy_order(sets: dict) -> tuple[list, list, list]:
    seen = {}
    remaining = sorted(sets)
    order = []
    gains = []
    singles = []
    while remaining:
        loc = max(remaining, key=lambda i: (count_seen(seen, sets[i], 0), count_seen(seen, sets[i], 1)))
        gains.append(count_seen(seen, sets[loc], 0))
        singles.append(count_seen(seen, sets[loc], 1))
        add_location(seen, sets[loc])
        remaining.remove(loc)
        order.append(loc)
    return order, gains, singles


def chao(seen: dict, m: int) -> float:
    n_obs = len(seen)
    f1 = sum(c == 1 for c in seen.values())
    f2 = sum(c == 2 for c in seen.values())
    return n_obs + (m - 1) / m * f1 * (f1 - 1) / (2 * (f2 + 1))


def chao_curve(sets: dict, order: list) -> tuple[list, list]:
    seen = {}
    n_observed = []
    n_chao = []
    for m, loc in enumerate(order, start=1):
        add_location(seen, sets[loc])
        n_observed.append(len(seen))
        n_chao.append(chao(seen, m))
    return n_observed, n_chao

In [ ]:
sets = {loc: set(g["identity"]) for loc, g in dataset.metadata.groupby("location")}
coords = dataset.metadata.groupby("location")[["longitude", "latitude"]].mean()
coords["region"] = np.where(coords["longitude"] < 16, "west", "east")

regions = [
    ("west", "Location 1: Šumava", {"Austria": [0.45, -0.12], "Czechia": [0.2, 0.2], "Germany": [-0.1, -0.1]}),
    ("east", "Location 2: Beskydy", {"Czechia": [-0.5, 0.1], "Poland": [0.05, 0.1], "Slovakia": [-0.1, -0.15]}),
]

curves = {}
for key, title, shifts in regions:
    locs = coords.index[coords["region"] == key]
    sets_region = {loc: sets[loc] for loc in locs}

    order, gains, singles = greedy_order(sets_region)
    coords.loc[order, "rank"] = np.arange(1, len(order) + 1)
    coords.loc[order, "gain_new"] = gains
    coords.loc[order, "gain_singles"] = singles

    curves[key] = chao_curve(sets_region, order)
    n_observed, n_chao = curves[key]
    print(f"{title}: {len(locs)} traps, {n_observed[-1]} observed, Chao {n_chao[-1]:.1f}")

In [ ]:
countries = gpd.read_file("/home/sadda/Projects/loc_earthranger/maps/ne_10m_admin_0_countries.shp")

pad = 0.1
n_clip = 5
vmax = coords["gain_new"].max()

fig, axd = plt.subplot_mosaic(
    [["west", "east", "east"], ["west", "chao_west", "chao_east"]],
    figsize=(17, 5.5),
    width_ratios=[2, 1, 1],
    height_ratios=[1, 0.5],
    gridspec_kw={"wspace": 0.25},
)
for key, title, shifts in regions:
    sub = coords[coords["region"] == key]
    ax = axd[key]
    ax.set_facecolor("lightblue")
    countries.plot(ax=ax, color="whitesmoke", edgecolor="black")

    zero = sub[(sub["gain_new"] == 0) & (sub["gain_singles"] == 0)]
    zero_singles = sub[(sub["gain_new"] == 0) & (sub["gain_singles"] > 0)]
    pos = sub[sub["gain_new"] > 0].sort_values("gain_new")
    ax.scatter(
        zero["longitude"], zero["latitude"], s=25, color="grey", marker="x", label="redundant (gain 0)", zorder=3
    )
    ax.scatter(
        zero_singles["longitude"],
        zero_singles["latitude"],
        color="red",
        marker="+",
        label="gain 0, re-sees singletons",
        zorder=3,
    )
    sc = ax.scatter(
        pos["longitude"],
        pos["latitude"],
        c=pos["gain_new"],
        s=30 + 15 * pos["gain_new"],
        cmap="viridis",
        vmin=0,
        vmax=vmax,
        edgecolor="black",
        zorder=3,
    )
    for loc, row in sub[sub["rank"] <= 10].iterrows():
        ax.annotate(
            str(int(row["rank"])),
            (row["longitude"], row["latitude"]),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=12,
            weight="bold",
            zorder=4,
        )

    lon_min = sub["longitude"].min() - pad
    lon_max = sub["longitude"].max() + pad
    lat_min = sub["latitude"].min() - pad
    lat_max = sub["latitude"].max() + pad

    visible = countries.clip(box(lon_min, lat_min, lon_max, lat_max))
    for _, country in visible.iterrows():
        if country.geometry.area < 0.02:
            continue
        name = country["NAME"]
        point = country.geometry.representative_point()
        shift = shifts.get(name, [0, 0])
        ax.text(
            point.x + shift[0],
            point.y + shift[1],
            name,
            ha="center",
            va="center",
            fontsize=14,
            color="blue",
            alpha=1,
            style="italic",
            zorder=2,
        )

    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_aspect(1 / np.cos(np.deg2rad(49)))
    ax.set_title(title)

    n_observed, n_chao = curves[key]
    chao_final = n_chao[-1]
    k = np.arange(1, len(n_chao) + 1)
    ax = axd[f"chao_{key}"]
    ax.set_title(title)
    ax.plot(k, n_chao, label="Chao estimate")
    ax.plot(k, n_observed, label="Observed")
    ax.axhspan(
        0.95 * chao_final,
        1.05 * chao_final,
        color="blue",
        alpha=0.15,
    )
    ax.set_xlim(n_clip, len(n_observed))
    ax.set_ylim(0, 1.1 * np.max(np.asarray(n_chao)[n_clip - 1 :]))
    ax.set_xlabel("Number of camera traps")
    if key == "west":
        ax.set_ylabel("Number of individuals")
    ax.grid(alpha=0.3)
    ax.legend()

fig.colorbar(sc, ax=list(axd.values()), label="New individuals added", shrink=0.8, panchor=False)
plt.savefig("camera_traps.png", bbox_inches="tight", dpi=300)